# Pair-Match Neighborhood Analysis (t-statistic)

This notebook analyzes matched-pair outputs from `op3_analysis.pair_match_neighborhoods` generated on the `t` layer.

Expected command:

```bash
python -m op3_analysis.pair_match_neighborhoods   --representation t   --output-dir results/pair_match_neighborhoods_t   --output-prefix pair_match_neighborhoods_t
```

Expected files:

- `pair_match_neighborhoods_t_truth_matches.csv`
- `pair_match_neighborhoods_t_truth_summary.csv`
- `pair_match_neighborhoods_t_detail.csv`
- `pair_match_neighborhoods_t_summary_by_cell_type.csv`
- `pair_match_neighborhoods_t_summary_overall.csv`

It focuses on:

1. coverage of matched truth pairs across dataset pairs,
2. continuous agreement on matched signatures in two scopes: DE-focused subsets and all shared genes,
3. DEG overlap across all threshold configs present in the result files,
4. neighborhood preservation (`overlap@k` and `kNN-edge Jaccard`) on the matched samples only,
5. pair-level, cell-type-level, and compound-level drilldowns.

The continuous and neighborhood views are expected to use `t` as the representation. The DEG panels read the precomputed DEG columns from the result files, so they reflect however the pipeline was run for DEG labeling and DEG-prediction overlap.


In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.options.display.max_columns = 300
sns.set_theme(context='talk', style='whitegrid')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'results').exists() and (PROJECT_ROOT.parent / 'results').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_SUBDIR = 'pair_match_neighborhoods_t'
PREFIX = 'pair_match_neighborhoods_t'
EXPECTED_REPRESENTATION = 't'
RESULTS_DIR = PROJECT_ROOT / 'results' / RESULTS_SUBDIR
if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f'Missing {RESULTS_DIR}. Generate t-stat results with: '
        f"python -m op3_analysis.pair_match_neighborhoods --representation {EXPECTED_REPRESENTATION} "
        f"--output-dir results/{RESULTS_SUBDIR} --output-prefix {PREFIX}"
    )
METADATA_PATH = RESULTS_DIR / f'{PREFIX}_metadata.json'
metadata = json.loads(METADATA_PATH.read_text()) if METADATA_PATH.exists() else {}

truth_matches = pd.read_csv(RESULTS_DIR / f'{PREFIX}_truth_matches.csv')
truth_summary = pd.read_csv(RESULTS_DIR / f'{PREFIX}_truth_summary.csv')
detail = pd.read_csv(RESULTS_DIR / f'{PREFIX}_detail.csv')
summary_by_cell_type = pd.read_csv(RESULTS_DIR / f'{PREFIX}_summary_by_cell_type.csv')
summary_overall = pd.read_csv(RESULTS_DIR / f'{PREFIX}_summary_overall.csv')

for frame in (truth_matches, truth_summary, detail, summary_by_cell_type, summary_overall):
    frame['pair'] = frame['query_dataset'] + ' -> ' + frame['db_dataset']
    frame['pair_key'] = frame['query_dataset'] + '__' + frame['db_dataset']

deg_configs = sorted({
    match.group(1)
    for col in summary_overall.columns
    for match in [re.match(r'mean_deg_signed_jaccard_(.+?)_weighted_mean$', col)]
    if match is not None
})
deg_subset_names = sorted({
    match.group(1)
    for col in summary_overall.columns
    for match in [re.match(r'(?:mean_deg_(?:signed|any)_jaccard_.+?__|mean_deg_pred_(?:signed|any)_jaccard_.+?__|query_loo_mean_deg_pred_(?:signed|any)_jaccard_.+?__|db_mean_deg_pred_(?:signed|any)_jaccard_.+?__)(de_top\d+_(?:query|db|union))_weighted_mean$', col)]
    if match is not None
})
neighbor_metrics = sorted({
    match.group(1)
    for col in summary_overall.columns
    for match in [re.match(r'mean_overlap_at_5_(?:de_top\d+_(?:query|db|union)_)?(pearson|cosine|mse)_weighted_mean$', col)]
    if match is not None
})
neighbor_subset_names = sorted({
    match.group(1)
    for col in summary_overall.columns
    for match in [re.match(r'mean_overlap_at_5_(de_top\d+_(?:query|db|union))_(pearson|cosine|mse)_weighted_mean$', col)]
    if match is not None
})
de_subset_names = sorted({
    match.group(1)
    for col in summary_overall.columns
    for match in [re.match(r'mean_match_(de_top\d+_(?:query|db|union))_pearson_weighted_mean$', col)]
    if match is not None
})


def summary_metric_col(base_col):
    return f'{base_col}_weighted_mean'


def match_metric_base(metric_name, subset_name=None):
    if subset_name:
        return f'match_{subset_name}_{metric_name}'
    return f'match_{metric_name}'


def match_metric_mean_base(metric_name, subset_name=None):
    if subset_name:
        return f'mean_match_{subset_name}_{metric_name}'
    return f'mean_match_{metric_name}'


def match_metric_col(metric_name, subset_name=None):
    return summary_metric_col(match_metric_mean_base(metric_name, subset_name))


def baseline_metric_base(metric_name, baseline_name='query_loo_mean', subset_name=None):
    if subset_name:
        return f'{baseline_name}_{subset_name}_{metric_name}'
    return f'{baseline_name}_{metric_name}'


def baseline_metric_col(metric_name, baseline_name='query_loo_mean', subset_name=None):
    return summary_metric_col(baseline_metric_base(metric_name, baseline_name, subset_name))


def nir_metric_base(baseline_name='query_loo_mean', subset_name=None):
    if subset_name:
        return f'mean_match_{subset_name}_nir_vs_{baseline_name}'
    return f'mean_match_nir_vs_{baseline_name}'


def nir_metric_col(baseline_name='query_loo_mean', subset_name=None):
    return summary_metric_col(nir_metric_base(baseline_name, subset_name))


def neighbor_overlap_base(k, metric_name, subset_name=None):
    if subset_name:
        return f'overlap_at_{int(k)}_{subset_name}_{metric_name}'
    return f'overlap_at_{int(k)}_{metric_name}'


def neighbor_overlap_mean_base(k, metric_name, subset_name=None):
    if subset_name:
        return f'mean_overlap_at_{int(k)}_{subset_name}_{metric_name}'
    return f'mean_overlap_at_{int(k)}_{metric_name}'


def neighbor_overlap_col(k, metric_name, subset_name=None):
    return summary_metric_col(neighbor_overlap_mean_base(k, metric_name, subset_name))


def neighbor_edge_base(k, metric_name, subset_name=None):
    if subset_name:
        return f'knn_edge_jaccard_at_{int(k)}_{subset_name}_{metric_name}'
    return f'knn_edge_jaccard_at_{int(k)}_{metric_name}'


def neighbor_edge_col(k, metric_name, subset_name=None):
    return summary_metric_col(neighbor_edge_base(k, metric_name, subset_name))


def deg_overlap_detail_base(kind_name, config_name, subset_name=None, predictor='match'):
    if predictor == 'match':
        base = f'deg_{kind_name}_jaccard_{config_name}'
    elif predictor == 'db':
        base = f'deg_pred_{kind_name}_jaccard_{config_name}'
    elif predictor in {'query_loo_mean', 'db_mean'}:
        base = f'{predictor}_deg_pred_{kind_name}_jaccard_{config_name}'
    else:
        raise ValueError(f'Unsupported DEG predictor: {predictor}')
    if subset_name:
        return f'{base}__{subset_name}'
    return base


def deg_overlap_mean_base(kind_name, config_name, subset_name=None, predictor='match'):
    detail_base = deg_overlap_detail_base(kind_name, config_name, subset_name=subset_name, predictor=predictor)
    if predictor in {'match', 'db'}:
        return f'mean_{detail_base}'
    return detail_base


def deg_overlap_col(kind_name, config_name, subset_name=None, predictor='match'):
    return summary_metric_col(deg_overlap_mean_base(kind_name, config_name, subset_name=subset_name, predictor=predictor))


def pretty_subset_name(subset_name):
    if subset_name is None:
        return 'all shared genes'
    return subset_name.replace('de_', '').replace('_', ' ')


def pretty_metric_name(metric_name):
    return {
        'pearson': 'Pearson',
        'cosine': 'cosine',
        'mse': 'MSE',
        'wmse': 'WMSE',
        'weighted_r2': 'weighted R^2',
    }.get(metric_name, metric_name)


def first_existing(candidates, frame=summary_overall):
    for candidate in candidates:
        if candidate is not None and candidate in frame.columns:
            return candidate
    return None


def metric_sort_ascending(metric_name):
    return metric_name in {'mse', 'wmse'}


def choose_metric_name(subset_name=None, frame=summary_overall):
    for metric_name in ('weighted_r2', 'pearson', 'wmse', 'mse'):
        candidate = match_metric_col(metric_name, subset_name)
        if candidate in frame.columns:
            return metric_name
    return None


def choose_neighbor_metric_name(subset_name=None, frame=summary_overall):
    for metric_name in ('pearson', 'cosine', 'mse'):
        candidate = neighbor_overlap_col(5, metric_name, subset_name)
        if candidate in frame.columns:
            return metric_name
    return None


def choose_de_subset(subset_names):
    for candidate in (
        'de_top50_union',
        'de_top20_union',
        'de_top50_query',
        'de_top50_db',
        'de_top20_query',
        'de_top20_db',
    ):
        if candidate in subset_names:
            return candidate
    return subset_names[0] if subset_names else None


def build_continuous_view(label, subset_name=None, preferred_metric=None, frame=summary_overall):
    metric_name = preferred_metric or choose_metric_name(subset_name, frame=frame)
    if metric_name is None:
        return None
    return {
        'label': label,
        'subset_name': subset_name,
        'metric_name': metric_name,
        'summary_col': match_metric_col(metric_name, subset_name),
        'mean_col': match_metric_mean_base(metric_name, subset_name),
        'detail_col': match_metric_base(metric_name, subset_name),
        'ascending': metric_sort_ascending(metric_name),
    }


def build_neighbor_view(label, subset_name=None, preferred_metric=None, frame=summary_overall):
    metric_name = (
        preferred_metric
        if preferred_metric is not None and neighbor_overlap_col(5, preferred_metric, subset_name) in frame.columns
        else choose_neighbor_metric_name(subset_name, frame=frame)
    )
    if metric_name is None:
        return None
    return {
        'label': label,
        'subset_name': subset_name,
        'metric_name': metric_name,
        'overlap_summary_cols': {
            int(k): neighbor_overlap_col(k, metric_name, subset_name)
            for k in (1, 5, 10)
        },
        'overlap_mean_cols': {
            int(k): neighbor_overlap_mean_base(k, metric_name, subset_name)
            for k in (1, 5, 10)
        },
        'overlap_detail_cols': {
            int(k): neighbor_overlap_base(k, metric_name, subset_name)
            for k in (1, 5, 10)
        },
        'edge_mean_col': neighbor_edge_base(5, metric_name, subset_name),
        'edge_summary_col': neighbor_edge_col(5, metric_name, subset_name),
    }


def describe_continuous_view(view):
    return f"{view['label']} ({pretty_subset_name(view['subset_name'])}, {pretty_metric_name(view['metric_name'])})"


def describe_neighbor_view(view):
    return f"{view['label']} ({pretty_subset_name(view['subset_name'])}, {pretty_metric_name(view['metric_name'])})"


FOCUS_DEG_SUBSET = choose_de_subset(deg_subset_names)

def choose_focus_deg_config(frame=summary_overall):
    if not deg_configs:
        return None
    best_cfg = None
    best_score = float('-inf')
    for cfg in deg_configs:
        for candidate_col in [
            deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='db') if FOCUS_DEG_SUBSET else None,
            deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='match') if FOCUS_DEG_SUBSET else None,
            deg_overlap_col('signed', cfg, predictor='db'),
            deg_overlap_col('signed', cfg, predictor='match'),
        ]:
            if candidate_col is None or candidate_col not in frame.columns:
                continue
            values = frame[candidate_col].to_numpy(dtype=np.float64)
            finite = np.isfinite(values)
            if not np.any(finite):
                continue
            score = float(np.nanmean(values[finite]))
            if score > best_score:
                best_score = score
                best_cfg = cfg
            break
    return best_cfg if best_cfg is not None else deg_configs[0]

FOCUS_DEG_CONFIG = choose_focus_deg_config()
FOCUS_NEIGHBOR_METRIC = 'pearson' if 'pearson' in neighbor_metrics else (neighbor_metrics[0] if neighbor_metrics else None)
DE_FOCUS_MATCH_SUBSET = choose_de_subset(de_subset_names)
DE_FOCUS_NEIGHBOR_SUBSET = choose_de_subset(neighbor_subset_names)
DE_FOCUS_VIEW = build_continuous_view('DE-focused', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET is not None else None
ALL_SHARED_VIEW = build_continuous_view('All shared genes', None)
TOP20_UNION_VIEW = build_continuous_view('DE top20 union', 'de_top20_union') if 'de_top20_union' in de_subset_names else None
ALL_NEIGHBOR_VIEW = build_neighbor_view('All shared genes neighborhood', None, FOCUS_NEIGHBOR_METRIC)
DE_NEIGHBOR_VIEW = build_neighbor_view('DE-focused neighborhood', DE_FOCUS_NEIGHBOR_SUBSET, FOCUS_NEIGHBOR_METRIC) if DE_FOCUS_NEIGHBOR_SUBSET is not None else None

DEFAULT_REPORT_COL = first_existing([
    DE_FOCUS_VIEW['summary_col'] if DE_FOCUS_VIEW else None,
    ALL_SHARED_VIEW['summary_col'] if ALL_SHARED_VIEW else None,
    'mean_match_pearson_weighted_mean',
])
DEFAULT_REPORT_ASCENDING = (
    DE_FOCUS_VIEW['ascending'] if DE_FOCUS_VIEW and DEFAULT_REPORT_COL == DE_FOCUS_VIEW['summary_col']
    else ALL_SHARED_VIEW['ascending'] if ALL_SHARED_VIEW and DEFAULT_REPORT_COL == ALL_SHARED_VIEW['summary_col']
    else False
)

print('Project root:', PROJECT_ROOT)
print('Results dir:', RESULTS_DIR)
print('Metadata representation:', metadata.get('representation', '<missing>'))
print('Metadata logfc_layer:', metadata.get('logfc_layer', '<missing>'))
print('Metadata pvalue_layer:', metadata.get('pvalue_layer', '<missing>'))
if metadata and metadata.get('representation') != EXPECTED_REPRESENTATION:
    print(f"Warning: expected representation {EXPECTED_REPRESENTATION!r}, found {metadata.get('representation')!r}")
print('DEG configs:', deg_configs)
print('DEG subset names:', deg_subset_names)
print('Neighborhood metrics:', neighbor_metrics)
print('Neighborhood subset names:', neighbor_subset_names)
print('DE subset names:', de_subset_names)
print('Focus DEG config:', FOCUS_DEG_CONFIG)
print('Focus DEG subset:', FOCUS_DEG_SUBSET)
print('Focus neighborhood metric:', FOCUS_NEIGHBOR_METRIC)
print('DE-focused view:', describe_continuous_view(DE_FOCUS_VIEW) if DE_FOCUS_VIEW else 'not available in current results')
print('All-shared view:', describe_continuous_view(ALL_SHARED_VIEW) if ALL_SHARED_VIEW else 'not available in current results')
print('DE-neighborhood view:', describe_neighbor_view(DE_NEIGHBOR_VIEW) if DE_NEIGHBOR_VIEW else 'not available in current results')
print('All-neighborhood view:', describe_neighbor_view(ALL_NEIGHBOR_VIEW) if ALL_NEIGHBOR_VIEW else 'not available in current results')


In [ ]:
overview = pd.DataFrame(
    {
        'table': [
            'truth_matches',
            'truth_summary',
            'detail',
            'summary_by_cell_type',
            'summary_overall',
        ],
        'rows': [
            len(truth_matches),
            len(truth_summary),
            len(detail),
            len(summary_by_cell_type),
            len(summary_overall),
        ],
    }
)
display(overview)

headline_cols = [
    'query_dataset',
    'db_dataset',
    'n_query_cell_types',
    'total_truth_queries',
]
for candidate_col in [
    DE_FOCUS_VIEW['summary_col'] if DE_FOCUS_VIEW else None,
    match_metric_col('wmse', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET else None,
    baseline_metric_col('wmse', 'query_loo_mean', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET else None,
    nir_metric_col('query_loo_mean', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET else None,
    ALL_SHARED_VIEW['summary_col'] if ALL_SHARED_VIEW else None,
    match_metric_col('wmse'),
    baseline_metric_col('wmse', 'query_loo_mean'),
    nir_metric_col('query_loo_mean'),
    deg_overlap_col('signed', FOCUS_DEG_CONFIG, predictor='match') if FOCUS_DEG_CONFIG else None,
    deg_overlap_col('signed', FOCUS_DEG_CONFIG, predictor='db') if FOCUS_DEG_CONFIG else None,
    deg_overlap_col('signed', FOCUS_DEG_CONFIG, subset_name=FOCUS_DEG_SUBSET, predictor='db') if FOCUS_DEG_CONFIG and FOCUS_DEG_SUBSET else None,
    deg_overlap_col('signed', FOCUS_DEG_CONFIG, subset_name=FOCUS_DEG_SUBSET, predictor='query_loo_mean') if FOCUS_DEG_CONFIG and FOCUS_DEG_SUBSET else None,
    f'mean_deg_signed_jaccard_{FOCUS_DEG_CONFIG}_weighted_mean' if FOCUS_DEG_CONFIG else None,
    f'mean_overlap_at_5_{FOCUS_NEIGHBOR_METRIC}_weighted_mean' if FOCUS_NEIGHBOR_METRIC else None,
    f'knn_edge_jaccard_at_5_{FOCUS_NEIGHBOR_METRIC}_weighted_mean' if FOCUS_NEIGHBOR_METRIC else None,
]:
    if candidate_col is not None and candidate_col in summary_overall.columns and candidate_col not in headline_cols:
        headline_cols.append(candidate_col)

sort_col = DEFAULT_REPORT_COL if DEFAULT_REPORT_COL in summary_overall.columns else 'total_truth_queries'
sort_ascending = DEFAULT_REPORT_ASCENDING if sort_col == DEFAULT_REPORT_COL else False

display(
    summary_overall[headline_cols]
    .sort_values(sort_col, ascending=sort_ascending)
    .reset_index(drop=True)
)


In [ ]:
def pair_pivot(df, value_col):
    matrix = df.pivot(index='query_dataset', columns='db_dataset', values=value_col)
    return matrix.sort_index().sort_index(axis=1)


def plot_pair_heatmap(df, value_col, title, fmt='.2f', cmap='viridis', center=None):
    if value_col not in df.columns:
        print(f'Skipping {title}: missing column {value_col}')
        return
    matrix = pair_pivot(df, value_col)
    fig_width = max(6, 1.5 * matrix.shape[1] + 2)
    fig_height = max(4, 1.0 * matrix.shape[0] + 2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sns.heatmap(
        matrix,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        center=center,
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'shrink': 0.8},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('DB dataset')
    ax.set_ylabel('Query dataset')
    plt.tight_layout()
    plt.show()


def overall_ranking(value_col, ascending=False):
    if value_col not in summary_overall.columns:
        return pd.DataFrame(columns=['pair', 'n_query_cell_types', 'total_truth_queries', value_col])
    cols = ['pair', 'n_query_cell_types', 'total_truth_queries', value_col]
    return summary_overall[cols].sort_values(value_col, ascending=ascending).reset_index(drop=True)


def top_cell_types_for_pair(query_dataset, db_dataset, metric_col, ascending=False, top_n=15):
    subset = summary_by_cell_type.loc[
        (summary_by_cell_type['query_dataset'] == query_dataset)
        & (summary_by_cell_type['db_dataset'] == db_dataset)
    ].copy()
    if metric_col not in subset.columns:
        return subset.head(0).reset_index(drop=True)
    return subset.sort_values(metric_col, ascending=ascending).head(top_n).reset_index(drop=True)


## Continuous Agreement

This notebook keeps two continuous views visible whenever the corresponding columns are present:

- `DE-focused`: top-DE subsets such as `de_top50_union`, scored with the calibrated metrics from the pipeline,
- `all shared genes`: the same matched rows scored across the full shared gene space.

In this t-statistic notebook, those continuous views are expected to come from result files generated with `--representation t`.


In [ ]:
plot_pair_heatmap(summary_overall, 'total_truth_queries', 'Total matched truth queries', fmt='.0f', cmap='Blues')
plot_pair_heatmap(summary_overall, 'n_query_cell_types', 'Matched query cell types', fmt='.0f', cmap='Greens')
plot_pair_heatmap(
    summary_overall,
    'total_shared_pubchem_cids_across_cell_types',
    'Total shared PubChem CIDs across matched cell types',
    fmt='.0f',
    cmap='Purples',
)

display(
    truth_summary[['query_dataset', 'db_dataset', 'query_cell_type', 'n_shared_pubchem_cids', 'n_truth_queries']]
    .sort_values('n_truth_queries', ascending=False)
    .head(20)
    .reset_index(drop=True)
)


## Matched-Signature Agreement

These are the continuous agreement metrics computed directly on the matched rows. The notebook keeps both reporting views visible: DE-focused subsets for primary reporting, and all shared genes for context.


In [ ]:
heatmap_specs = []

if DE_FOCUS_VIEW is not None:
    heatmap_specs.append(
        {
            'value_col': DE_FOCUS_VIEW['summary_col'],
            'title': f"Weighted mean {pretty_subset_name(DE_FOCUS_VIEW['subset_name'])} {pretty_metric_name(DE_FOCUS_VIEW['metric_name'])}",
            'fmt': '.2f',
            'cmap': 'coolwarm',
            'center': 0 if not DE_FOCUS_VIEW['ascending'] else None,
        }
    )
    de_wmse_col = match_metric_col('wmse', DE_FOCUS_MATCH_SUBSET)
    if de_wmse_col in summary_overall.columns and de_wmse_col != DE_FOCUS_VIEW['summary_col']:
        heatmap_specs.append(
            {
                'value_col': de_wmse_col,
                'title': f"Weighted mean {pretty_subset_name(DE_FOCUS_MATCH_SUBSET)} WMSE (lower is better)",
                'fmt': '.2f',
                'cmap': 'mako_r',
                'center': None,
            }
        )

if ALL_SHARED_VIEW is not None:
    heatmap_specs.append(
        {
            'value_col': ALL_SHARED_VIEW['summary_col'],
            'title': f"Weighted mean {pretty_subset_name(None)} {pretty_metric_name(ALL_SHARED_VIEW['metric_name'])}",
            'fmt': '.2f',
            'cmap': 'coolwarm',
            'center': 0 if not ALL_SHARED_VIEW['ascending'] else None,
        }
    )
    all_wmse_col = match_metric_col('wmse')
    if all_wmse_col in summary_overall.columns and all_wmse_col != ALL_SHARED_VIEW['summary_col']:
        heatmap_specs.append(
            {
                'value_col': all_wmse_col,
                'title': 'Weighted mean all-shared-gene WMSE (lower is better)',
                'fmt': '.2f',
                'cmap': 'mako_r',
                'center': None,
            }
        )

for spec in heatmap_specs:
    plot_pair_heatmap(
        summary_overall,
        spec['value_col'],
        spec['title'],
        fmt=spec['fmt'],
        cmap=spec['cmap'],
        center=spec['center'],
    )

if TOP20_UNION_VIEW is not None:
    plot_pair_heatmap(
        summary_overall,
        TOP20_UNION_VIEW['summary_col'],
        f"Weighted mean {pretty_subset_name(TOP20_UNION_VIEW['subset_name'])} {pretty_metric_name(TOP20_UNION_VIEW['metric_name'])}",
        fmt='.2f',
        cmap='coolwarm',
        center=0 if not TOP20_UNION_VIEW['ascending'] else None,
    )

if DE_FOCUS_VIEW is not None:
    print('Top pairs by DE-focused continuous metric')
    display(overall_ranking(DE_FOCUS_VIEW['summary_col'], ascending=DE_FOCUS_VIEW['ascending']).head(12))
if ALL_SHARED_VIEW is not None:
    print('Top pairs by all-shared-gene continuous metric')
    display(overall_ranking(ALL_SHARED_VIEW['summary_col'], ascending=ALL_SHARED_VIEW['ascending']).head(12))


## DEG Overlap Across All Threshold Grids

The DEG section in this notebook shows every DEG threshold config present in the result files, not only one focus threshold.

Two related views are reported:

- `thresholded actual overlap`: query and matched-DB DEG labels are compared directly from the precomputed DEG columns,
- `DEG prediction overlap`: the query DEG labels are treated as truth, and the matched DB profile or mean baseline is converted into a size-matched up/down prediction using the precomputed DEG-prediction columns.

When DE subset DEG columns are present, the notebook also shows the default DEG subset focus (typically `de_top50_union`) for every threshold.


In [ ]:
deg_threshold_rows = []
for cfg in deg_configs:
    row = {'deg_config': cfg}
    candidate_map = {
        'actual_all_signed': deg_overlap_col('signed', cfg, predictor='match'),
        'actual_all_any': deg_overlap_col('any', cfg, predictor='match'),
        'pred_all_signed': deg_overlap_col('signed', cfg, predictor='db'),
        'query_baseline_all_signed': deg_overlap_col('signed', cfg, predictor='query_loo_mean'),
        'db_baseline_all_signed': deg_overlap_col('signed', cfg, predictor='db_mean'),
        'actual_focus_subset_signed': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='match') if FOCUS_DEG_SUBSET else None,
        'pred_focus_subset_signed': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='db') if FOCUS_DEG_SUBSET else None,
        'query_baseline_focus_subset_signed': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='query_loo_mean') if FOCUS_DEG_SUBSET else None,
        'db_baseline_focus_subset_signed': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='db_mean') if FOCUS_DEG_SUBSET else None,
    }
    available = False
    for key, col in candidate_map.items():
        if col is not None and col in summary_overall.columns:
            row[key] = float(summary_overall[col].mean())
            available = True
        else:
            row[key] = np.nan
    if available:
        deg_threshold_rows.append(row)

deg_threshold_summary = pd.DataFrame(deg_threshold_rows)
if deg_threshold_summary.empty:
    print('No DEG overlap columns are present in the current result CSVs.')
else:
    deg_sort_col = first_existing([
        'pred_focus_subset_signed',
        'actual_focus_subset_signed',
        'pred_all_signed',
        'actual_all_signed',
    ], frame=deg_threshold_summary)
    if deg_sort_col is None:
        deg_sort_col = deg_threshold_summary.columns[-1]
    deg_threshold_summary = deg_threshold_summary.sort_values(deg_sort_col, ascending=False).reset_index(drop=True)
    display(deg_threshold_summary)

    deg_plot_cols = [
        col for col in [
            'actual_all_signed',
            'pred_all_signed',
            'query_baseline_all_signed',
            'actual_focus_subset_signed',
            'pred_focus_subset_signed',
            'query_baseline_focus_subset_signed',
        ]
        if col in deg_threshold_summary.columns and deg_threshold_summary[col].notna().any()
    ]
    if deg_plot_cols:
        deg_long = deg_threshold_summary.melt(
            id_vars='deg_config',
            value_vars=deg_plot_cols,
            var_name='view',
            value_name='score',
        )
        fig_height = max(5, 0.6 * len(deg_threshold_summary) + 2)
        fig, ax = plt.subplots(figsize=(14, fig_height))
        sns.barplot(data=deg_long, x='score', y='deg_config', hue='view', ax=ax)
        ax.set_title('Mean DEG Jaccard across pairs by threshold rule')
        ax.set_xlabel('Mean Jaccard across dataset pairs')
        ax.set_ylabel('DEG config')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

    for cfg in deg_configs:
        print(f'DEG threshold: {cfg}')
        heatmap_specs = [
            {
                'value_col': deg_overlap_col('signed', cfg, predictor='match'),
                'title': f'Signed DEG Jaccard, all genes: {cfg}',
                'cmap': 'YlOrBr',
            },
            {
                'value_col': deg_overlap_col('signed', cfg, predictor='db'),
                'title': f'Signed DEG prediction Jaccard, all genes: {cfg}',
                'cmap': 'flare',
            },
            {
                'value_col': deg_overlap_col('signed', cfg, predictor='query_loo_mean'),
                'title': f'Query mean baseline signed DEG Jaccard, all genes: {cfg}',
                'cmap': 'crest',
            },
        ]
        if FOCUS_DEG_SUBSET is not None:
            heatmap_specs.extend([
                {
                    'value_col': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='match'),
                    'title': f'Signed DEG Jaccard, {pretty_subset_name(FOCUS_DEG_SUBSET)}: {cfg}',
                    'cmap': 'YlGnBu',
                },
                {
                    'value_col': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='db'),
                    'title': f'Signed DEG prediction Jaccard, {pretty_subset_name(FOCUS_DEG_SUBSET)}: {cfg}',
                    'cmap': 'rocket',
                },
                {
                    'value_col': deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='query_loo_mean'),
                    'title': f'Query mean baseline signed DEG Jaccard, {pretty_subset_name(FOCUS_DEG_SUBSET)}: {cfg}',
                    'cmap': 'viridis',
                },
            ])
        for spec in heatmap_specs:
            if spec['value_col'] in summary_overall.columns:
                plot_pair_heatmap(summary_overall, spec['value_col'], spec['title'], fmt='.2f', cmap=spec['cmap'])

        ranking_specs = [
            ('actual signed overlap, all genes', deg_overlap_col('signed', cfg, predictor='match')),
            ('predicted signed overlap, all genes', deg_overlap_col('signed', cfg, predictor='db')),
        ]
        if FOCUS_DEG_SUBSET is not None:
            ranking_specs.extend([
                (
                    f'actual signed overlap, {pretty_subset_name(FOCUS_DEG_SUBSET)}',
                    deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='match'),
                ),
                (
                    f'predicted signed overlap, {pretty_subset_name(FOCUS_DEG_SUBSET)}',
                    deg_overlap_col('signed', cfg, subset_name=FOCUS_DEG_SUBSET, predictor='db'),
                ),
            ])
        for label, col in ranking_specs:
            if col in summary_overall.columns:
                print(f'Top pairs by {label} | {cfg}')
                display(overall_ranking(col, ascending=False).head(12))


## Neighborhood Preservation

These metrics compare the neighborhood structure inside the matched samples only.

- `all shared genes`: the existing neighborhood view over the full shared-gene space,
- `DE-focused`: the new neighborhood view over anchor-specific DE subsets such as `de_top50_union`, `de_top20_query`, and `de_top20_db`.

The DE-subset neighborhood metrics use the anchor row's selected genes to rank neighbors within the matched samples of each dataset, then compare those directed neighbor sets across datasets.


In [ ]:
neighbor_rows = []
for view in [ALL_NEIGHBOR_VIEW, DE_NEIGHBOR_VIEW]:
    if view is None:
        continue
    overlap5_col = view['overlap_summary_cols'][5]
    if overlap5_col not in summary_overall.columns:
        continue
    row = {
        'scope': view['label'],
        'subset_name': pretty_subset_name(view['subset_name']),
        'neighborhood_metric': view['metric_name'],
        'mean_overlap_at_1_across_pairs': summary_overall[view['overlap_summary_cols'][1]].mean() if view['overlap_summary_cols'][1] in summary_overall.columns else np.nan,
        'mean_overlap_at_5_across_pairs': summary_overall[view['overlap_summary_cols'][5]].mean(),
        'mean_overlap_at_10_across_pairs': summary_overall[view['overlap_summary_cols'][10]].mean() if view['overlap_summary_cols'][10] in summary_overall.columns else np.nan,
        'mean_knn_edge_jaccard_at_5_across_pairs': summary_overall[view['edge_summary_col']].mean() if view['edge_summary_col'] in summary_overall.columns else np.nan,
    }
    row['scope_label'] = f"{view['label']} | {view['metric_name']}"
    neighbor_rows.append(row)

neighbor_summary = pd.DataFrame(neighbor_rows)
if not neighbor_summary.empty:
    neighbor_summary = neighbor_summary.sort_values('mean_overlap_at_5_across_pairs', ascending=False).reset_index(drop=True)
display(neighbor_summary)

if not neighbor_summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.barplot(data=neighbor_summary, x='mean_overlap_at_5_across_pairs', y='scope_label', color='seagreen', ax=axes[0])
    axes[0].set_title('Mean overlap@5 across pairs')
    axes[0].set_xlabel('overlap@5')
    axes[0].set_ylabel('Neighborhood view')
    sns.barplot(data=neighbor_summary, x='mean_knn_edge_jaccard_at_5_across_pairs', y='scope_label', color='mediumpurple', ax=axes[1])
    axes[1].set_title('Mean kNN-edge Jaccard@5 across pairs')
    axes[1].set_xlabel('edge Jaccard@5')
    axes[1].set_ylabel('')
    plt.tight_layout()
    plt.show()

for view in [ALL_NEIGHBOR_VIEW, DE_NEIGHBOR_VIEW]:
    if view is None:
        continue
    for k in (1, 5, 10):
        overlap_col = view['overlap_summary_cols'][k]
        if overlap_col in summary_overall.columns:
            plot_pair_heatmap(
                summary_overall,
                overlap_col,
                f"Neighborhood overlap@{k}: {view['label']} ({view['metric_name']})",
                fmt='.2f',
                cmap='crest',
            )
    if view['edge_summary_col'] in summary_overall.columns:
        plot_pair_heatmap(
            summary_overall,
            view['edge_summary_col'],
            f"kNN-edge Jaccard@5: {view['label']} ({view['metric_name']})",
            fmt='.2f',
            cmap='magma',
        )
    if view['overlap_summary_cols'][5] in summary_overall.columns:
        print(f"Top pairs by {view['label'].lower()} overlap@5")
        display(overall_ranking(view['overlap_summary_cols'][5], ascending=False).head(12))


## Cell-Type Drilldown

Select the strongest pair under the primary continuous metric and inspect which cell types drive the result. The drilldown keeps both continuous views and, when available, both neighborhood views side by side.


In [ ]:
focus_sort_view = DE_FOCUS_VIEW if DE_FOCUS_VIEW is not None else ALL_SHARED_VIEW
focus_sort_col = focus_sort_view['summary_col'] if focus_sort_view is not None else 'total_truth_queries'
focus_sort_ascending = focus_sort_view['ascending'] if focus_sort_view is not None else False

focus_pair_row = summary_overall.sort_values(focus_sort_col, ascending=focus_sort_ascending).iloc[0]
FOCUS_QUERY_DATASET = focus_pair_row['query_dataset']
FOCUS_DB_DATASET = focus_pair_row['db_dataset']
print('Focus pair:', FOCUS_QUERY_DATASET, '->', FOCUS_DB_DATASET)

pair_celltypes = top_cell_types_for_pair(
    FOCUS_QUERY_DATASET,
    FOCUS_DB_DATASET,
    metric_col=focus_sort_view['mean_col'] if focus_sort_view is not None else 'mean_match_pearson',
    ascending=focus_sort_ascending,
    top_n=20,
)

display_cols = [
    'query_cell_type',
    'n_truth_queries',
]
for candidate_col in [
    DE_FOCUS_VIEW['mean_col'] if DE_FOCUS_VIEW else None,
    match_metric_mean_base('wmse', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET else None,
    ALL_SHARED_VIEW['mean_col'] if ALL_SHARED_VIEW else None,
    match_metric_mean_base('wmse'),
    match_metric_mean_base('pearson', 'de_top20_union') if 'de_top20_union' in de_subset_names else None,
    deg_overlap_mean_base('signed', FOCUS_DEG_CONFIG, predictor='match') if FOCUS_DEG_CONFIG else None,
    deg_overlap_mean_base('signed', FOCUS_DEG_CONFIG, predictor='db') if FOCUS_DEG_CONFIG else None,
    deg_overlap_mean_base('signed', FOCUS_DEG_CONFIG, subset_name=FOCUS_DEG_SUBSET, predictor='db') if FOCUS_DEG_CONFIG and FOCUS_DEG_SUBSET else None,
    deg_overlap_mean_base('signed', FOCUS_DEG_CONFIG, subset_name=FOCUS_DEG_SUBSET, predictor='query_loo_mean') if FOCUS_DEG_CONFIG and FOCUS_DEG_SUBSET else None,
    ALL_NEIGHBOR_VIEW['overlap_mean_cols'][5] if ALL_NEIGHBOR_VIEW else None,
    ALL_NEIGHBOR_VIEW['edge_mean_col'] if ALL_NEIGHBOR_VIEW else None,
    DE_NEIGHBOR_VIEW['overlap_mean_cols'][5] if DE_NEIGHBOR_VIEW else None,
    DE_NEIGHBOR_VIEW['edge_mean_col'] if DE_NEIGHBOR_VIEW else None,
    f'mean_deg_signed_jaccard_{FOCUS_DEG_CONFIG}' if FOCUS_DEG_CONFIG else None,
]:
    if candidate_col is not None and candidate_col in pair_celltypes.columns and candidate_col not in display_cols:
        display_cols.append(candidate_col)
display(pair_celltypes[display_cols])

plot_specs = []
if DE_FOCUS_VIEW is not None and DE_FOCUS_VIEW['mean_col'] in pair_celltypes.columns:
    plot_specs.append(
        {
            'x': DE_FOCUS_VIEW['mean_col'],
            'title': f"{pretty_subset_name(DE_FOCUS_VIEW['subset_name'])} {pretty_metric_name(DE_FOCUS_VIEW['metric_name'])}",
            'xlabel': f"Mean {pretty_metric_name(DE_FOCUS_VIEW['metric_name'])}",
            'color': 'steelblue',
        }
    )
if ALL_SHARED_VIEW is not None and ALL_SHARED_VIEW['mean_col'] in pair_celltypes.columns:
    plot_specs.append(
        {
            'x': ALL_SHARED_VIEW['mean_col'],
            'title': f"{pretty_subset_name(None)} {pretty_metric_name(ALL_SHARED_VIEW['metric_name'])}",
            'xlabel': f"Mean {pretty_metric_name(ALL_SHARED_VIEW['metric_name'])}",
            'color': 'darkorange',
        }
    )
if ALL_NEIGHBOR_VIEW is not None and ALL_NEIGHBOR_VIEW['overlap_mean_cols'][5] in pair_celltypes.columns:
    plot_specs.append(
        {
            'x': ALL_NEIGHBOR_VIEW['overlap_mean_cols'][5],
            'title': f"{ALL_NEIGHBOR_VIEW['label']} overlap@5 ({ALL_NEIGHBOR_VIEW['metric_name']})",
            'xlabel': 'Mean overlap@5',
            'color': 'seagreen',
        }
    )
if DE_NEIGHBOR_VIEW is not None and DE_NEIGHBOR_VIEW['overlap_mean_cols'][5] in pair_celltypes.columns:
    plot_specs.append(
        {
            'x': DE_NEIGHBOR_VIEW['overlap_mean_cols'][5],
            'title': f"{DE_NEIGHBOR_VIEW['label']} overlap@5 ({DE_NEIGHBOR_VIEW['metric_name']})",
            'xlabel': 'Mean overlap@5',
            'color': 'mediumpurple',
        }
    )

if plot_specs:
    fig, axes = plt.subplots(1, len(plot_specs), figsize=(6 * len(plot_specs), 8), sharey=True)
    if len(plot_specs) == 1:
        axes = [axes]
    for ax, spec in zip(axes, plot_specs):
        sns.barplot(data=pair_celltypes, y='query_cell_type', x=spec['x'], color=spec['color'], ax=ax)
        ax.set_title(spec['title'])
        ax.set_xlabel(spec['xlabel'])
        ax.set_ylabel('Query cell type' if ax is axes[0] else '')
    plt.tight_layout()
    plt.show()

neighbor_scatter_x = ALL_NEIGHBOR_VIEW['overlap_mean_cols'][5] if ALL_NEIGHBOR_VIEW and ALL_NEIGHBOR_VIEW['overlap_mean_cols'][5] in summary_by_cell_type.columns else None
neighbor_scatter_y = DE_NEIGHBOR_VIEW['overlap_mean_cols'][5] if DE_NEIGHBOR_VIEW and DE_NEIGHBOR_VIEW['overlap_mean_cols'][5] in summary_by_cell_type.columns else None
if neighbor_scatter_x is not None and neighbor_scatter_y is not None and neighbor_scatter_x != neighbor_scatter_y:
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.scatterplot(
        data=summary_by_cell_type,
        x=neighbor_scatter_x,
        y=neighbor_scatter_y,
        hue='query_dataset',
        style='db_dataset',
        size='n_truth_queries',
        sizes=(40, 400),
        alpha=0.85,
        ax=ax,
    )
    ax.set_title('Cell-type level all-gene vs DE-focused neighborhood overlap@5')
    ax.set_xlabel(f"{ALL_NEIGHBOR_VIEW['label']} overlap@5 ({ALL_NEIGHBOR_VIEW['metric_name']})")
    ax.set_ylabel(f"{DE_NEIGHBOR_VIEW['label']} overlap@5 ({DE_NEIGHBOR_VIEW['metric_name']})")
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


## Detail-Level Distribution for One Pair and Cell Type

Use the strongest cell type within the focus pair to inspect per-matched-row distributions. The detail table includes both the DE-focused and all-shared-gene continuous scores, plus both neighborhood views when they are present.


In [ ]:
FOCUS_CELL_TYPE = pair_celltypes.iloc[0]['query_cell_type']
focus_detail = detail.loc[
    (detail['query_dataset'] == FOCUS_QUERY_DATASET)
    & (detail['db_dataset'] == FOCUS_DB_DATASET)
    & (detail['query_cell_type'] == FOCUS_CELL_TYPE)
].copy()

print('Focus cell type:', FOCUS_CELL_TYPE)
print('Matched rows:', len(focus_detail))

detail_cols = [
    'pubchem_cid',
    'pert_time_h',
    'pert_dose_uM',
]
for candidate_col in [
    DE_FOCUS_VIEW['detail_col'] if DE_FOCUS_VIEW else None,
    match_metric_base('wmse', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET else None,
    ALL_SHARED_VIEW['detail_col'] if ALL_SHARED_VIEW else None,
    match_metric_base('wmse'),
    'match_de_top20_query_pearson',
    'match_de_top20_db_pearson',
    'match_de_top20_union_pearson',
    'match_de_top20_union_wmse',
    'match_de_top50_union_n_genes',
    deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, predictor='match') if FOCUS_DEG_CONFIG else None,
    deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, predictor='db') if FOCUS_DEG_CONFIG else None,
    deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, subset_name=FOCUS_DEG_SUBSET, predictor='db') if FOCUS_DEG_CONFIG and FOCUS_DEG_SUBSET else None,
    deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, subset_name=FOCUS_DEG_SUBSET, predictor='query_loo_mean') if FOCUS_DEG_CONFIG and FOCUS_DEG_SUBSET else None,
    ALL_NEIGHBOR_VIEW['overlap_detail_cols'][5] if ALL_NEIGHBOR_VIEW else None,
    DE_NEIGHBOR_VIEW['overlap_detail_cols'][5] if DE_NEIGHBOR_VIEW else None,
    f'deg_signed_jaccard_{FOCUS_DEG_CONFIG}' if FOCUS_DEG_CONFIG else None,
]:
    if candidate_col is not None and candidate_col in focus_detail.columns and candidate_col not in detail_cols:
        detail_cols.append(candidate_col)
display(focus_detail[detail_cols].head(20))

hist_specs = []
if DE_FOCUS_VIEW is not None and DE_FOCUS_VIEW['detail_col'] in focus_detail.columns:
    hist_specs.append(
        {
            'col': DE_FOCUS_VIEW['detail_col'],
            'title': f"{pretty_subset_name(DE_FOCUS_VIEW['subset_name'])} {pretty_metric_name(DE_FOCUS_VIEW['metric_name'])}",
            'xlabel': pretty_metric_name(DE_FOCUS_VIEW['metric_name']),
            'color': 'steelblue',
        }
    )
if ALL_SHARED_VIEW is not None and ALL_SHARED_VIEW['detail_col'] in focus_detail.columns and ALL_SHARED_VIEW['detail_col'] != (DE_FOCUS_VIEW['detail_col'] if DE_FOCUS_VIEW else None):
    hist_specs.append(
        {
            'col': ALL_SHARED_VIEW['detail_col'],
            'title': f"{pretty_subset_name(None)} {pretty_metric_name(ALL_SHARED_VIEW['metric_name'])}",
            'xlabel': pretty_metric_name(ALL_SHARED_VIEW['metric_name']),
            'color': 'darkorange',
        }
    )

de_wmse_detail_col = match_metric_base('wmse', DE_FOCUS_MATCH_SUBSET) if DE_FOCUS_MATCH_SUBSET else None
if de_wmse_detail_col is not None and de_wmse_detail_col in focus_detail.columns:
    hist_specs.append(
        {
            'col': de_wmse_detail_col,
            'title': f"{pretty_subset_name(DE_FOCUS_MATCH_SUBSET)} WMSE",
            'xlabel': 'WMSE',
            'color': 'firebrick',
        }
    )

all_overlap_detail_col = ALL_NEIGHBOR_VIEW['overlap_detail_cols'][5] if ALL_NEIGHBOR_VIEW else None
if all_overlap_detail_col is not None and all_overlap_detail_col in focus_detail.columns:
    hist_specs.append(
        {
            'col': all_overlap_detail_col,
            'title': f"{ALL_NEIGHBOR_VIEW['label']} overlap@5 ({ALL_NEIGHBOR_VIEW['metric_name']})",
            'xlabel': 'overlap@5',
            'color': 'seagreen',
        }
    )

de_overlap_detail_col = DE_NEIGHBOR_VIEW['overlap_detail_cols'][5] if DE_NEIGHBOR_VIEW else None
if de_overlap_detail_col is not None and de_overlap_detail_col in focus_detail.columns:
    hist_specs.append(
        {
            'col': de_overlap_detail_col,
            'title': f"{DE_NEIGHBOR_VIEW['label']} overlap@5 ({DE_NEIGHBOR_VIEW['metric_name']})",
            'xlabel': 'overlap@5',
            'color': 'mediumpurple',
        }
    )

if hist_specs:
    fig, axes = plt.subplots(1, len(hist_specs), figsize=(6 * len(hist_specs), 5))
    if len(hist_specs) == 1:
        axes = [axes]
    for ax, spec in zip(axes, hist_specs):
        sns.histplot(focus_detail[spec['col']], bins=30, kde=True, ax=ax, color=spec['color'])
        ax.set_title(spec['title'])
        ax.set_xlabel(spec['xlabel'])
    plt.tight_layout()
    plt.show()

if all_overlap_detail_col is not None and de_overlap_detail_col is not None and all_overlap_detail_col in focus_detail.columns and de_overlap_detail_col in focus_detail.columns and all_overlap_detail_col != de_overlap_detail_col:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=focus_detail,
        x=all_overlap_detail_col,
        y=de_overlap_detail_col,
        hue=deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, predictor='db') if FOCUS_DEG_CONFIG and deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, predictor='db') in focus_detail.columns else (f'deg_signed_jaccard_{FOCUS_DEG_CONFIG}' if FOCUS_DEG_CONFIG and f'deg_signed_jaccard_{FOCUS_DEG_CONFIG}' in focus_detail.columns else None),
        palette='viridis',
        ax=ax,
    )
    ax.set_title(f'{FOCUS_QUERY_DATASET} -> {FOCUS_DB_DATASET} | {FOCUS_CELL_TYPE}')
    ax.set_xlabel(f"{ALL_NEIGHBOR_VIEW['label']} overlap@5 ({ALL_NEIGHBOR_VIEW['metric_name']})")
    ax.set_ylabel(f"{DE_NEIGHBOR_VIEW['label']} overlap@5 ({DE_NEIGHBOR_VIEW['metric_name']})")
    plt.tight_layout()
    plt.show()
elif DE_FOCUS_VIEW is not None and DE_FOCUS_VIEW['detail_col'] in focus_detail.columns and all_overlap_detail_col is not None and all_overlap_detail_col in focus_detail.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=focus_detail,
        x=DE_FOCUS_VIEW['detail_col'],
        y=all_overlap_detail_col,
        hue=deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, predictor='db') if FOCUS_DEG_CONFIG and deg_overlap_detail_base('signed', FOCUS_DEG_CONFIG, predictor='db') in focus_detail.columns else (f'deg_signed_jaccard_{FOCUS_DEG_CONFIG}' if FOCUS_DEG_CONFIG and f'deg_signed_jaccard_{FOCUS_DEG_CONFIG}' in focus_detail.columns else None),
        palette='viridis',
        ax=ax,
    )
    ax.set_title(f'{FOCUS_QUERY_DATASET} -> {FOCUS_DB_DATASET} | {FOCUS_CELL_TYPE}')
    ax.set_xlabel(f"{pretty_subset_name(DE_FOCUS_VIEW['subset_name'])} {pretty_metric_name(DE_FOCUS_VIEW['metric_name'])}")
    ax.set_ylabel(f"{ALL_NEIGHBOR_VIEW['label']} overlap@5 ({ALL_NEIGHBOR_VIEW['metric_name']})")
    plt.tight_layout()
    plt.show()


## Baselines

These tables use the precomputed mean-predictor baselines from the matched-pair pipeline:

- `query leave-one-out mean baseline`: predict each row with the mean of the other perturbations in the same query dataset and cell type,
- `db mean baseline`: predict each row with the mean perturbation profile in the DB dataset and cell type,
- `NIR`: normalized improvement ratio over the chosen mean baseline, computed from WMSE as `1 - matched_wmse / baseline_wmse`.

The notebook reports baseline comparisons for the DE-focused view and, when available, the all-shared-gene view. Older result files will not have these precomputed columns yet; in that case the section prints a reminder instead of failing.


In [ ]:
summary_with_baselines = summary_overall.copy()


def build_baseline_view(label, subset_name=None):
    wmse_col = match_metric_col('wmse', subset_name)
    query_baseline_wmse_col = baseline_metric_col('wmse', 'query_loo_mean', subset_name)
    db_baseline_wmse_col = baseline_metric_col('wmse', 'db_mean', subset_name)
    nir_query_col = nir_metric_col('query_loo_mean', subset_name)
    nir_db_col = nir_metric_col('db_mean', subset_name)
    metric_name = choose_metric_name(subset_name, frame=summary_with_baselines)
    metric_col = match_metric_col(metric_name, subset_name) if metric_name is not None else None
    query_baseline_metric_col = baseline_metric_col(metric_name, 'query_loo_mean', subset_name) if metric_name is not None else None
    if wmse_col not in summary_with_baselines.columns or query_baseline_wmse_col not in summary_with_baselines.columns:
        return None
    return {
        'label': label,
        'subset_name': subset_name,
        'metric_name': metric_name,
        'metric_col': metric_col,
        'query_baseline_metric_col': query_baseline_metric_col,
        'wmse_col': wmse_col,
        'query_baseline_wmse_col': query_baseline_wmse_col,
        'db_baseline_wmse_col': db_baseline_wmse_col,
        'nir_query_col': nir_query_col,
        'nir_db_col': nir_db_col,
    }


baseline_views = []
if DE_FOCUS_MATCH_SUBSET is not None:
    de_baseline_view = build_baseline_view('DE-focused', DE_FOCUS_MATCH_SUBSET)
    if de_baseline_view is not None:
        baseline_views.append(de_baseline_view)
all_baseline_view = build_baseline_view('All shared genes', None)
if all_baseline_view is not None:
    baseline_views.append(all_baseline_view)

if not baseline_views:
    print('Baseline columns are not present in the current result CSVs. Rerun op3_analysis.pair_match_neighborhoods to populate the DE-focused and all-gene baseline summaries.')
else:
    baseline_display_cols = ['pair', 'total_truth_queries']
    for view in baseline_views:
        for candidate_col in [
            view['metric_col'],
            view['query_baseline_metric_col'],
            view['wmse_col'],
            view['query_baseline_wmse_col'],
            view['db_baseline_wmse_col'],
            view['nir_query_col'],
            view['nir_db_col'],
        ]:
            if candidate_col is not None and candidate_col in summary_with_baselines.columns and candidate_col not in baseline_display_cols:
                baseline_display_cols.append(candidate_col)
    primary_view = baseline_views[0]
    display(
        summary_with_baselines[baseline_display_cols]
        .sort_values(primary_view['nir_query_col'], ascending=False)
        .reset_index(drop=True)
    )

    fig, axes = plt.subplots(1, len(baseline_views), figsize=(7 * len(baseline_views), 6))
    if len(baseline_views) == 1:
        axes = [axes]
    for ax, view in zip(axes, baseline_views):
        sns.scatterplot(
            data=summary_with_baselines,
            x=view['query_baseline_wmse_col'],
            y=view['wmse_col'],
            hue='query_dataset',
            style='db_dataset',
            size='total_truth_queries',
            sizes=(40, 350),
            alpha=0.85,
            ax=ax,
        )
        low = min(summary_with_baselines[view['query_baseline_wmse_col']].min(), summary_with_baselines[view['wmse_col']].min())
        high = max(summary_with_baselines[view['query_baseline_wmse_col']].max(), summary_with_baselines[view['wmse_col']].max())
        ax.plot([low, high], [low, high], linestyle='--', color='black', linewidth=1)
        ax.set_title(f"{view['label']} WMSE vs query mean baseline")
        ax.set_xlabel('Query mean baseline WMSE')
        ax.set_ylabel('Matched-pair WMSE')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    for view in baseline_views:
        if view['nir_query_col'] in summary_with_baselines.columns:
            plot_pair_heatmap(
                summary_with_baselines,
                view['nir_query_col'],
                f"NIR vs query mean baseline: {view['label'].lower()} ({pretty_subset_name(view['subset_name'])})",
                fmt='.2f',
                cmap='coolwarm',
                center=0,
            )

    celltype_with_baselines = summary_by_cell_type.copy()
    primary_celltype_cols = ['pair', 'query_cell_type', 'n_truth_queries']
    for view in baseline_views:
        for candidate_col in [
            match_metric_mean_base(view['metric_name'], view['subset_name']) if view['metric_name'] is not None else None,
            baseline_metric_base(view['metric_name'], 'query_loo_mean', view['subset_name']) if view['metric_name'] is not None else None,
            match_metric_mean_base('wmse', view['subset_name']),
            baseline_metric_base('wmse', 'query_loo_mean', view['subset_name']),
            nir_metric_base('query_loo_mean', view['subset_name']),
        ]:
            if candidate_col is not None and candidate_col in celltype_with_baselines.columns and candidate_col not in primary_celltype_cols:
                primary_celltype_cols.append(candidate_col)
    display(
        celltype_with_baselines.loc[
            (celltype_with_baselines['query_dataset'] == FOCUS_QUERY_DATASET)
            & (celltype_with_baselines['db_dataset'] == FOCUS_DB_DATASET),
            primary_celltype_cols,
        ]
        .sort_values(primary_celltype_cols[-1], ascending=False)
        .reset_index(drop=True)
    )
